O codigo a seguir vai no portal da transparencia e baixa o csv do ano especificado para viagens a serviço 
extrai os dados e cria um dataframe de cada arquivo para trabalhar em cima deles

In [28]:
import requests
import os

#constante
ano = 2024



def download_viagens_csv(ano: int, caminho_destino: str):
    """
    Baixa o CSV de viagens do Portal da Transparência para o ano especificado.
    Exemplo de uso:
        download_viagens_csv(ano, 'viagens_{ano}.csv')
    """

    url = f"https://portaldatransparencia.gov.br/download-de-dados/viagens/{ano}"
    print(f"🔄 Fazendo download de: {url}")

    # Envia requisição GET
    resposta = requests.get(url)
    try:
        resposta.raise_for_status()
    except requests.HTTPError:
        raise SystemExit(f"Erro ao acessar URL {url}: {resposta.status_code}")

    # O site redireciona para um link direto do CSV, então precisamos acompanhar esse redirect
    url_csv = resposta.url
    print(f"➡️ Redirecionado para o CSV: {url_csv}")

    # Baixa o conteúdo do CSV
    conteudo = requests.get(url_csv)
    try:
        conteudo.raise_for_status()
    except requests.HTTPError:
        raise SystemExit(f"Erro ao baixar o CSV: {conteudo.status_code}")

    # Salva o conteúdo no caminho indicado
    with open(caminho_destino, "wb") as f:
        f.write(conteudo.content)

    print(f"✅ CSV salvo em: {caminho_destino}")




nome_arquivo = f"dadosViagens/viagens_{ano}.zip"

# Garante que o diretório existe antes de salvar o arquivo
os.makedirs(os.path.dirname(nome_arquivo), exist_ok=True)

download_viagens_csv(ano, nome_arquivo)


🔄 Fazendo download de: https://portaldatransparencia.gov.br/download-de-dados/viagens/2024
➡️ Redirecionado para o CSV: https://dadosabertos-download.cgu.gov.br/PortalDaTransparencia/saida/viagens/2024_20250713_Viagens.zip
✅ CSV salvo em: dadosViagens/viagens_2024.zip


In [29]:
import zipfile
import os

def descompactar_arquivo_zip(caminho_zip: str, destino: str = "./"):
    """
    Descompacta um arquivo .zip para o diretório de destino.

    Args:
        caminho_zip (str): Caminho para o arquivo .zip.
        destino (str): Diretório onde os arquivos serão extraídos (padrão: pasta atual).
    """
    try:
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
            zip_ref.extractall(destino)
            print(f"✅ Arquivos extraídos para: {os.path.abspath(destino)}")
            print("📁 Conteúdo extraído:")
            for nome_arquivo in zip_ref.namelist():
                print(f" - {nome_arquivo}")
    except zipfile.BadZipFile:
        print("❌ O arquivo ZIP está corrompido ou inválido.")
    except FileNotFoundError:
        print("❌ Arquivo .zip não encontrado.")
    except Exception as e:
        print(f"❌ Erro ao descompactar: {e}")



caminho_zip = f"dadosViagens/viagens_{ano}.zip" 
pasta_destino = f"dadosViagens/dados_viagens{ano}"  

descompactar_arquivo_zip(caminho_zip, pasta_destino)


try:
    os.remove(caminho_zip)
    print(f"✅ Arquivo ZIP removido: {caminho_zip}")
except FileNotFoundError:
    print(f"❌ Arquivo ZIP não encontrado para remoção: {caminho_zip}")
except Exception as e:
    print(f"❌ Erro ao remover o arquivo ZIP: {e}")


✅ Arquivos extraídos para: d:\projetoExtensao\dadosViagens\dados_viagens2024
📁 Conteúdo extraído:
 - 2024_Pagamento.csv
 - 2024_Passagem.csv
 - 2024_Trecho.csv
 - 2024_Viagem.csv
✅ Arquivo ZIP removido: dadosViagens/viagens_2024.zip


In [30]:
import pandas as pd
import os

pasta_base = "dadosViagens/dados_viagens"  


def carregar_csvs_viagens_por_ano(ano: int, pasta_base: str ):
    """
    Lê os arquivos CSV de um ano específico, usando a primeira linha como cabeçalho e removendo-a dos dados.
    
    Args:
        ano: Ano desejado (ex: 2024).
        pasta_base: Pasta onde os dados estão salvos (sem o número do ano).
    
    Returns:
        Três DataFrames: passagem_df, trecho_df, viagem_df
    """

    nomes = {
        "passagem_df": f"{ano}_Passagem.csv",
        "trecho_df": f"{ano}_Trecho.csv",
        "viagem_df": f"{ano}_Viagem.csv"
    }

    pasta_dados = os.path.join(pasta_base + str(ano))
    resultados = {}

    for var_name, nome_arquivo in nomes.items():
        caminho_completo = os.path.join(pasta_dados, nome_arquivo)
        try:
            # Lê o arquivo inteiro sem considerar cabeçalhos
            df_raw = pd.read_csv(caminho_completo, sep=";", encoding="latin1", header=None, low_memory=False)

            # Usa a primeira linha como nomes das colunas
            df_raw.columns = df_raw.iloc[0]
            
            # Remove a primeira linha (que virou cabeçalho)
            df = df_raw.drop(index=0).reset_index(drop=True)

            resultados[var_name] = df
            print(f"✅ {nome_arquivo} carregado com {len(df)} linhas (sem cabeçalho).")
        except FileNotFoundError:
            print(f"❌ Arquivo não encontrado: {nome_arquivo}")
            resultados[var_name] = None
        except Exception as e:
            print(f"❌ Erro ao carregar {nome_arquivo}: {e}")
            resultados[var_name] = None

    return resultados.get("passagem_df"), resultados.get("trecho_df"), resultados.get("viagem_df")




passagem_df, trecho_df, viagem_df = carregar_csvs_viagens_por_ano(ano,pasta_base)


✅ 2024_Passagem.csv carregado com 399305 linhas (sem cabeçalho).
✅ 2024_Trecho.csv carregado com 1703677 linhas (sem cabeçalho).
✅ 2024_Viagem.csv carregado com 791520 linhas (sem cabeçalho).


In [31]:
# Exibe os dados
if passagem_df is not None:
    print("\n🔍 Primeiras linhas de PASSAGEM:")
    print(passagem_df.head())


🔍 Primeiras linhas de PASSAGEM:
0 Identificador do processo de viagem Número da Proposta (PCDP)  \
0                 0000000000019177818              000001/24-1C   
1                 0000000000019177818              000001/24-1C   
2                 0000000000019220977              000001/24-1C   
3                 0000000000019255612                 000002/24   
4                 0000000000019255728                 000001/24   

0 Meio de transporte País - Origem ida   UF - Origem ida Cidade - Origem ida  \
0              Aéreo            Brasil  Distrito Federal            Brasília   
1              Aéreo            França               NaN               Paris   
2              Aéreo            Brasil            Paraná            Curitiba   
3              Aéreo          Alemanha               NaN             Munique   
4              Aéreo          Alemanha               NaN             Munique   

0 País - Destino ida  UF - Destino ida Cidade - Destino ida  \
0             França

In [20]:

if trecho_df is not None:
    print("\n🔍 Primeiras linhas de TRECHO:")
    print(trecho_df.head())


🔍 Primeiras linhas de TRECHO:
0 Identificador do processo de viagem  Número da Proposta (PCDP)  \
0                  0000000000018831091                 000011/24   
1                  0000000000018831091                 000011/24   
2                  0000000000018831495                 000001/24   
3                  0000000000018831495                 000001/24   
4                  0000000000018831777                 000002/24   

0 Sequência Trecho Origem - Data Origem - País   Origem - UF Origem - Cidade  \
0                2    25/02/2024        Brasil          Acre      Rio Branco   
1                1    23/02/2024        Brasil          Acre          Xapuri   
2                2    22/01/2024        Brasil     São Paulo       São Paulo   
3                1    18/01/2024        Brasil  Minas Gerais      Uberlândia   
4                2    04/03/2024        Brasil     São Paulo       São Paulo   

0 Destino - Data Destino - País  Destino - UF Destino - Cidade  \
0     25/02/2

In [21]:

if viagem_df is not None:
    print("\n🔍 Primeiras linhas de VIAGEM:")
    print(viagem_df.head())



🔍 Primeiras linhas de VIAGEM:
0 Identificador do processo de viagem Número da Proposta (PCDP)   Situação  \
0                 0000000000018831091                 000011/24  Realizada   
1                 0000000000018831495                 000001/24  Realizada   
2                 0000000000018831777                 000002/24  Realizada   
3                 0000000000018831821                 000003/24  Realizada   
4                 0000000000019177818              000001/24-1C  Realizada   

0 Viagem Urgente Justificativa Urgência Viagem Código do órgão superior  \
0            NÃO                Sem informação                    26000   
1            NÃO                Sem informação                    26000   
2            NÃO                Sem informação                    26000   
3            NÃO                Sem informação                    26000   
4            SIM         Inclusão das diárias.                    26000   

0  Nome do órgão superior Código órgão solicitant

In [22]:
print(passagem_df)

0      Identificador do processo de viagem Número da Proposta (PCDP)  \
0                      0000000000019177818              000001/24-1C   
1                      0000000000019177818              000001/24-1C   
2                      0000000000019220977              000001/24-1C   
3                      0000000000019255612                 000002/24   
4                      0000000000019255728                 000001/24   
...                                    ...                       ...   
399300                 0000000002024001938             Sem informaçã   
399301                 0000000002024001946             Sem informaçã   
399302                 0000000002024001946             Sem informaçã   
399303                 0000000002024001948             Sem informaçã   
399304                 0000000002024001948             Sem informaçã   

0      Meio de transporte País - Origem ida    UF - Origem ida  \
0                   Aéreo            Brasil   Distrito Federal   
1  

In [23]:
print(trecho_df)

0       Identificador do processo de viagem  Número da Proposta (PCDP)  \
0                        0000000000018831091                 000011/24   
1                        0000000000018831091                 000011/24   
2                        0000000000018831495                 000001/24   
3                        0000000000018831495                 000001/24   
4                        0000000000018831777                 000002/24   
...                                      ...                       ...   
1703672                  0000000002024001938             Sem informaçã   
1703673                  0000000002024001946             Sem informaçã   
1703674                  0000000002024001946             Sem informaçã   
1703675                  0000000002024001948             Sem informaçã   
1703676                  0000000002024001948             Sem informaçã   

0       Sequência Trecho Origem - Data Origem - País        Origem - UF  \
0                      2    25/02/20

In [24]:
print(viagem_df)

0      Identificador do processo de viagem Número da Proposta (PCDP)  \
0                      0000000000018831091                 000011/24   
1                      0000000000018831495                 000001/24   
2                      0000000000018831777                 000002/24   
3                      0000000000018831821                 000003/24   
4                      0000000000019177818              000001/24-1C   
...                                    ...                       ...   
791515                 0000000002024001911             Sem informaçã   
791516                 0000000002024001912             Sem informaçã   
791517                 0000000002024001938             Sem informaçã   
791518                 0000000002024001946             Sem informaçã   
791519                 0000000002024001948             Sem informaçã   

0            Situação Viagem Urgente Justificativa Urgência Viagem  \
0           Realizada            NÃO                Sem informaçã

In [25]:
trecho_df.describe()

,Identificador do processo de viagem,Número da Proposta (PCDP),Sequência Trecho,Origem - Data,Origem - País,Origem - UF,Origem - Cidade,Destino - Data,Destino - País,Destino - UF,Destino - Cidade,Meio de transporte,Número Diárias,Missao?
count,1703677,1703677,1703677,1703677,1703677,1669287,1703677,1703677,1703677,1669475,1703677,1703677,1703677,1703677
unique,673228,107224,72,448,177,28,6478,453,178,28,6476,8,400,2
top,0000000000019617576,Sem informaçã,1,25/11/2024,Brasil,São Paulo,Brasília,29/11/2024,Brasil,São Paulo,Brasília,Veículo Oficial,"0,50",Não
freq,72,2083,673228,10565,1669266,183929,174541,16988,1669454,184645,175832,848096,577169,860690


In [26]:
passagem_df.describe()

,Identificador do processo de viagem,Número da Proposta (PCDP),Meio de transporte,País - Origem ida,UF - Origem ida,Cidade - Origem ida,País - Destino ida,UF - Destino ida,Cidade - Destino ida,País - Origem volta,UF - Origem volta,Cidade - Origem volta,Pais - Destino volta,UF - Destino volta,Cidade - Destino volta,Valor da passagem,Taxa de serviço,Data da emissão/compra,Hora da emissão/compra
count,399305,399305,399305,399305,387312,399305,399305,384923,399305,399305,395765,399305,399305,398420,399305,399305,399305,397603,399305
unique,221141,56739,4,156,28,1090,161,28,1166,127,28,610,107,28,343,179610,2345,505,1200
top,0000000000019668216,Sem informaçã,Aéreo,Brasil,Distrito Federal,Brasília,Brasil,Distrito Federal,Brasília,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"0,00","0,00",18/10/2024,00:00
freq,37,1702,392504,387311,112421,112421,384922,113360,113360,366613,366613,366613,366613,366613,366613,919,361153,2471,1743


In [27]:
viagem_df.describe()

,Identificador do processo de viagem,Número da Proposta (PCDP),Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Função,Descrição Função,Período - Data de início,Período - Data de fim,Destinos,Motivo,Valor diárias,Valor passagens,Valor devolução,Valor outros gastos
count,791520,791520,791520,791520,791297,791520,791520,791520,791520,660320,...,791520,791517,791520,791520,791520,791519,791520,791520,791520,791520
unique,791519,107225,2,2,168784,27,26,175,175,155682,...,174,156,366,457,35522,361666,72858,159688,4679,16254
top,0000000000020598362,Informações p,Realizada,SIM,Sem informação,26000,Ministério da Educação,-1,Sem informação,***.000.000-**,...,-1,Sem informação,25/11/2024,29/11/2024,Informações protegidas por sigilo,Informação protegida por sigilo nos termos da ...,"0,00","0,00","0,00","0,00"
freq,2,118291,780827,462015,211501,160247,160247,150651,150651,933,...,360545,360545,8290,8533,118291,118291,91640,544606,765972,755956


Tira todos os Nan(not a number) do dataframe 

In [33]:
passagem_df.dropna()

,Identificador do processo de viagem,Número da Proposta (PCDP),Meio de transporte,País - Origem ida,UF - Origem ida,Cidade - Origem ida,País - Destino ida,UF - Destino ida,Cidade - Destino ida,País - Origem volta,UF - Origem volta,Cidade - Origem volta,Pais - Destino volta,UF - Destino volta,Cidade - Destino volta,Valor da passagem,Taxa de serviço,Data da emissão/compra,Hora da emissão/compra
21,0000000000019292022,000019/24,Aéreo,Brasil,Paraná,Foz do Iguaçu,Brasil,Distrito Federal,Brasília,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"602,65","0,00",11/10/2023,21:34
22,0000000000019292144,000020/24,Aéreo,Brasil,Paraná,Foz do Iguaçu,Brasil,Distrito Federal,Brasília,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"602,65","0,00",11/10/2023,16:25
26,0000000000019327561,000001/24,Aéreo,Brasil,São Paulo,São Paulo,Brasil,Distrito Federal,Brasília,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"1921,16","0,00",25/09/2023,16:43
28,0000000000019327561,000001/24,Aéreo,Brasil,Distrito Federal,Brasília,Brasil,São Paulo,São Paulo,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"1,00","0,00",25/09/2023,16:43
32,0000000000019334971,000001/24-1C,Aéreo,Brasil,Distrito Federal,Brasília,Brasil,Santa Catarina,Jaguaruna,Brasil,Santa Catarina,Jaguaruna,Brasil,Distrito Federal,Brasília,"1827,40","0,00",13/11/2023,11:35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397597,0000000000020771980,022789/24-1C,Rodoviário,Brasil,Amazonas,Manaus,Brasil,Roraima,Boa Vista,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"338,79","0,00",05/03/2025,17:59
397598,0000000000020772010,022788/24,Rodoviário,Brasil,Ceará,Fortaleza,Brasil,Ceará,Crateús,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"100,16","0,00",05/03/2025,16:09
397599,0000000000020775328,022790/24,Rodoviário,Brasil,Roraima,Boa Vista,Brasil,Amazonas,Manaus,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,Sem informação,"338,79","0,00",05/03/2025,16:11
397600,0000000000020778579,002228/24,Aéreo,Brasil,Minas Gerais,Belo Horizonte,Brasil,Distrito Federal,Brasília,Brasil,Distrito Federal,Brasília,Brasil,Minas Gerais,Belo Horizonte,"4250,24","0,01",25/02/2025,11:21


In [36]:
trecho_df.dropna()

,Identificador do processo de viagem,Número da Proposta (PCDP),Sequência Trecho,Origem - Data,Origem - País,Origem - UF,Origem - Cidade,Destino - Data,Destino - País,Destino - UF,Destino - Cidade,Meio de transporte,Número Diárias,Missao?
0,0000000000018831091,000011/24,2,25/02/2024,Brasil,Acre,Rio Branco,25/02/2024,Brasil,Acre,Xapuri,Veículo Próprio,"0,50",Não
1,0000000000018831091,000011/24,1,23/02/2024,Brasil,Acre,Xapuri,25/02/2024,Brasil,Acre,Rio Branco,Veículo Próprio,"2,00",Sim
2,0000000000018831495,000001/24,2,22/01/2024,Brasil,São Paulo,São Paulo,22/01/2024,Brasil,Minas Gerais,Uberlândia,Aéreo,"0,00",Não
3,0000000000018831495,000001/24,1,18/01/2024,Brasil,Minas Gerais,Uberlândia,22/01/2024,Brasil,São Paulo,São Paulo,Aéreo,"0,00",Sim
4,0000000000018831777,000002/24,2,04/03/2024,Brasil,São Paulo,São Paulo,04/03/2024,Brasil,Minas Gerais,Uberlândia,Aéreo,"0,00",Não
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1703672,0000000002024001938,Sem informaçã,2,27/12/2024,Brasil,Distrito Federal,Brasília,27/12/2024,Brasil,São Paulo,São Paulo,Aéreo,"0,00",Não
1703673,0000000002024001946,Sem informaçã,1,29/12/2024,Brasil,Rio Grande do Sul,Passo Fundo,31/12/2024,Brasil,Distrito Federal,Brasília,Aéreo,"0,00",Não
1703674,0000000002024001946,Sem informaçã,2,31/12/2024,Brasil,Distrito Federal,Brasília,31/12/2024,Brasil,Rio Grande do Sul,Santo Ângelo,Aéreo,"0,00",Não
1703675,0000000002024001948,Sem informaçã,1,30/12/2024,Brasil,São Paulo,São Paulo,30/12/2024,Brasil,Distrito Federal,Brasília,Aéreo,"0,00",Não


In [37]:
viagem_df.dropna()

,Identificador do processo de viagem,Número da Proposta (PCDP),Situação,Viagem Urgente,Justificativa Urgência Viagem,Código do órgão superior,Nome do órgão superior,Código órgão solicitante,Nome órgão solicitante,CPF viajante,...,Função,Descrição Função,Período - Data de início,Período - Data de fim,Destinos,Motivo,Valor diárias,Valor passagens,Valor devolução,Valor outros gastos
0,0000000000018831091,000011/24,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26425,"Instituto Federal de Educação, Ciência e Tecno...",***.405.257-**,...,-1,Sem informação,23/02/2024,25/02/2024,Rio Branco/AC,Ministrar a disciplina de Conceitos e Aplicaçõ...,"929,18","0,00","0,00","0,00"
1,0000000000018831495,000001/24,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26274,Fundação Universidade Federal de Uberlândia,***.587.016-**,...,-1,Sem informação,18/01/2024,22/01/2024,São Paulo/SP,Participação nas atividades do ESTAGIO DE CAPA...,"0,00","0,00","0,00","0,00"
2,0000000000018831777,000002/24,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26274,Fundação Universidade Federal de Uberlândia,***.587.016-**,...,-1,Sem informação,29/02/2024,04/03/2024,São Paulo/SP,Participação nas atividades do ESTAGIO DE CAPA...,"0,00","0,00","0,00","0,00"
3,0000000000018831821,000003/24,Realizada,NÃO,Sem informação,26000,Ministério da Educação,26274,Fundação Universidade Federal de Uberlândia,***.587.016-**,...,-1,Sem informação,11/04/2024,15/04/2024,São Paulo/SP,Participação nas atividades do ESTAGIO DE CAPA...,"0,00","0,00","0,00","0,00"
4,0000000000019177818,000001/24-1C,Realizada,SIM,Inclusão das diárias.,26000,Ministério da Educação,26271,Fundação Universidade de Brasília,***.626.601-**,...,-1,Sem informação,29/01/2024,06/02/2024,"Lille/França, Paris/França",Programação dos eventos que a Professora irá p...,"14176,38","6892,31","0,00","0,00"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
791514,0000000002024001906,Sem informaçã,Realizada,NÃO,Sem informação,-1,Sem informação,-1,Sem informação,***.759.868-**,...,-1,Sem informação,18/12/2024,19/12/2024,Sem informação,Cumprir agenda de trabalho em Brasília.,"1109,09","3180,06","0,00","0,00"
791515,0000000002024001911,Sem informaçã,Realizada,NÃO,Sem informação,-1,Sem informação,-1,Sem informação,***.902.438-**,...,-1,Sem informação,16/12/2024,16/12/2024,Sem informação,"Participar, como palestrante, no evento '1 ano...","262,05","0,00","0,00","0,00"
791516,0000000002024001912,Sem informaçã,Realizada,NÃO,Sem informação,-1,Sem informação,-1,Sem informação,***.697.505-**,...,-1,Sem informação,17/12/2024,18/12/2024,Sem informação,Assessorar o Diretor de Fiscalização e cumprir...,"1204,09","4044,69","0,00","0,00"
791517,0000000002024001938,Sem informaçã,Não realizada,NÃO,Sem informação,-1,Sem informação,-1,Sem informação,***.827.438-**,...,-1,Sem informação,26/12/2024,27/12/2024,Sem informação,Cumprir agenda em Brasília.,"0,00","0,00","0,00","0,00"
